# 08 — Identity Continuity Analysis

Quantitative tools for characterizing identity coupling topologies:
- VQE attractor basin (energy gap = robustness)
- Coherence budget (max circuit depth on Heron r2)
- Entanglement witness (CHSH S-parameter)
- Quantum fingerprint (spectral + SHA-256)
- Orchestrator phase mapping (18 ↔ 35 oscillators)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from scpn_quantum_control.identity.binding_spec import (
    ORCHESTRATOR_MAPPING,
    orchestrator_to_quantum_phases,
    quantum_to_orchestrator_phases,
)
from scpn_quantum_control.identity.coherence_budget import coherence_budget, fidelity_at_depth
from scpn_quantum_control.identity.entanglement_witness import disposition_entanglement_map
from scpn_quantum_control.identity.ground_state import IdentityAttractor
from scpn_quantum_control.identity.identity_key import identity_fingerprint

## 1. Identity Attractor Basin

The robustness gap $E_1 - E_0$ measures how stable the identity topology is against perturbations.

In [ ]:
# Use 4-qubit subsystem (full 18-qubit binding spec is intractable on statevector)
from scpn_quantum_control.bridge.knm_hamiltonian import OMEGA_N_16, build_knm_paper27

K4 = build_knm_paper27(L=4)
omega4 = OMEGA_N_16[:4]
attractor = IdentityAttractor(K4, omega4, ansatz_reps=2)
result = attractor.solve(maxiter=200, seed=42)
print(f"Ground energy: {result['ground_energy']:.6f}")
print(f"Robustness gap: {result['robustness_gap']:.4f}")
print(f"Eigenvalues: {result['eigenvalues'][:4]}")
print()
print("Note: full 18-oscillator spec requires 2^18 = 262K dim statevector.")
print("Use hardware or tensor network methods for full-scale identity analysis.")

## 2. Coherence Budget

Maximum circuit depth before fidelity drops below threshold on Heron r2 hardware.

In [ ]:
depths = np.arange(5, 200, 5)
fidelities_4q = [fidelity_at_depth(d, n_qubits=4) for d in depths]
fidelities_8q = [fidelity_at_depth(d, n_qubits=8) for d in depths]
fidelities_16q = [fidelity_at_depth(d, n_qubits=16) for d in depths]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(depths, fidelities_4q, label="4 qubits")
ax.plot(depths, fidelities_8q, label="8 qubits")
ax.plot(depths, fidelities_16q, label="16 qubits")
ax.axhline(0.5, color="red", linestyle="--", alpha=0.5, label="F=0.5 threshold")
ax.set_xlabel("Circuit depth")
ax.set_ylabel("Fidelity")
ax.set_title("Coherence Budget: Heron r2")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for n in [4, 8, 16]:
    b = coherence_budget(n_qubits=n, fidelity_threshold=0.5)
    print(f"  {n}q: max depth = {b['max_depth']}")

## 3. Entanglement Witness

CHSH S-parameter for qubit pairs. $S > 2$ certifies entanglement.

In [ ]:
from scpn_quantum_control.bridge.knm_hamiltonian import OMEGA_N_16, build_knm_paper27
from scpn_quantum_control.phase.phase_vqe import PhaseVQE

K = build_knm_paper27(L=4)
omega = OMEGA_N_16[:4]
vqe = PhaseVQE(K, omega, ansatz_reps=2)
vqe.solve(maxiter=200, seed=42)
sv = vqe.ground_state()

emap = disposition_entanglement_map(sv, disposition_labels=["L1", "L2", "L3", "L4"])
print(f"Entangled pairs: {emap['n_entangled']} / {emap['n_pairs']}")
print(f"Integration metric: {emap['integration_metric']:.4f}")
for p in emap["pairs"]:
    print(f"  {p['label_a']}-{p['label_b']}: S={p['S']:.4f} {'*' if p['entangled'] else ''}")

## 4. Quantum Fingerprint

In [ ]:
fp = identity_fingerprint(K, omega, maxiter=100)
print(f"Fiedler value: {fp['spectral']['fiedler']:.4f}")
print(f"Commitment: {fp['commitment'][:32]}...")
print(f"Spectral keys: {list(fp['spectral'].keys())}")

## 5. Orchestrator Phase Mapping

18 quantum oscillators map to 35 identity_coherence domainpack oscillators.

In [ ]:
theta_q = np.linspace(0, 2 * np.pi, 18, endpoint=False)
orch = quantum_to_orchestrator_phases(theta_q)
theta_back = orchestrator_to_quantum_phases(orch)
roundtrip_err = np.max(np.abs(np.angle(np.exp(1j * (theta_back - theta_q)))))

print(f"Quantum oscillators: {len(theta_q)}")
print(f"Orchestrator oscillators: {len(orch)}")
print(f"Roundtrip max error: {roundtrip_err:.2e}")
print("\nMapping summary:")
for qid, orch_ids in list(ORCHESTRATOR_MAPPING.items())[:6]:
    print(f"  {qid} -> {orch_ids}")